# stars

> Mask stars in a Euclid image.

In [ ]:
# | default_exp euclid.stars

In [ ]:
# | exporti

from warnings import catch_warnings, filterwarnings

import numpy as np
from photutils.segmentation import detect_sources
from photutils.utils import circular_footprint
from nicl.mask import fft_binary_erosion

In [ ]:
# | export


def star_mask(bitmask, instrument, verbose=False):
    """Create a star mask from the DQ/FLG bitmask.

    This is only applicable to individual calibrated frames.
    """
    if verbose:
        print("Creating star mask")
    if instrument == "VIS":
        # use the STARSIGNAL flag
        mask = bitmask & 2**18 > 0
        erosion_radius = 20
        footprint = circular_footprint(radius=erosion_radius)
        mask = fft_binary_erosion(mask, structure=footprint)
        growth_factor = 0.5
    else:
        # identify stars as saturated pixels that are not otherwise invalid (ignoring persistence)
        mask = bitmask & 2**10 > 0
        if verbose:
            print(f"Number of pixels with saturation masked: {mask.sum()}")
        for b in [2, 3, 4, 6, 7, 8, 9, 16, 23]:
            mask &= bitmask & 2**b == 0
            if verbose:
                print(
                    f"Number of pixels masked after checking bitmask {b}: {mask.sum()}"
                )
        growth_factor = 15
    min_pixels = 10
    star_mask = np.zeros(mask.shape, dtype=bool)
    if mask.sum() >= min_pixels:
        # segment the sources and dilate proportionally to the size of the source
        with catch_warnings():
            filterwarnings("ignore", ".*No sources were found*")
            star_segm = detect_sources(mask.astype(float), 0.5, npixels=min_pixels)
        if star_segm is not None:
            if verbose:
                print(f"Found {len(star_segm.segments)} stars")
            areas = np.unique(star_segm.areas)
            for area in areas:
                if verbose:
                    print(f"Masking stars with area {area}")
                idx = np.where(star_segm.areas == area)[0]
                labels = star_segm.labels[idx]
                segm = star_segm.copy()
                segm.keep_labels(labels)
                footprint = circular_footprint(
                    radius=int(growth_factor * np.sqrt(area))
                )
                star_mask |= segm.make_source_mask(footprint=footprint)
    return star_mask